# KG1 — EVAL 086 + candidato no full947 (Claude) — Rota B
**Run all.** Colab A100 + Secret HF_KEY. Mede ACC REAL (métrica oficial vLLM, rank<=32) do 086 e de um candidato no full947 canônico (947). 086 (0.86)=piso. Para um candidato diverso, defina `os.environ['CAND_REPO']` antes do run.

In [ ]:
import os,subprocess,sys
print('[1/4] clone repo branch', flush=True)
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'],check=True)
os.chdir('/content/kg1')
print('[2/4] deps base (+vllm)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','vllm==0.19.1','safetensors','huggingface_hub','hf_xet','einops','ninja'],check=False)
import torch
py=f"cp{sys.version_info.major}{sys.version_info.minor}"
tmm='.'.join(torch.__version__.split('+')[0].split('.')[:2])
cu='cu'+((torch.version.cuda or '12').split('.')[0])
abi='TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
print(f'[env] {py} torch{tmm} {cu} cxx11abi{abi}', flush=True)
print('[3/4] mamba-ssm 2.3.1 (WHEEL PRONTO ~10s; fallback source)', flush=True)
url=f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{tmm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
print('  tentando wheel:', url, flush=True)
if subprocess.run([sys.executable,'-m','pip','install','--no-deps',url]).returncode!=0:
    print('  >>> wheel nao casou -> compilando do source', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'],check=False)
print('[4/4] causal-conv1d 1.6.1 (source ~5min)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'],check=False)
print('DEPS OK',flush=True)


In [ ]:
from google.colab import userdata
import os
for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
    try:
        v=userdata.get(k)
        if v: os.environ['HF_TOKEN']=v; os.environ['HF_KEY']=v; break
    except Exception: pass
assert os.environ.get('HF_TOKEN'),'Defina HF_KEY no Colab Secrets'
print('HF token OK',flush=True)


In [ ]:
import os,subprocess,sys,time,glob
os.chdir('/content/kg1')
from huggingface_hub import snapshot_download
SOL='/content/kg1/artifacts/v1244_cot_safe/full947_solution.csv'  # canonico 947 (metrica oficial)
# --- LIVE-LOG: stream eval -> HF (Claude monitora AO VIVO) ---
os.environ['KG1_LIVE_LOG_HF_REPO']='felipesp1983/kg1-live-logs'
os.environ['KG1_LIVE_LOG_HF_REPO_TYPE']='dataset'
os.environ.setdefault('KG1_REQUIRE_LIVE_LOG_UPLOAD','0')   # eval longo: hiccup de upload NAO aborta

def find_adapter_dirs(root):
    # acha TODA pasta com adapter_config.json: raiz (086) OU runs/<id>/{final,checkpoint-N} (candidato)
    return sorted({os.path.dirname(c) for c in glob.glob(root+'/**/adapter_config.json', recursive=True)})

def run_eval(tag, adapter_dir):
    out='/content/eval_'+tag
    os.environ['RUN_ID']='v1244_eval_'+tag+'_'+time.strftime('%Y%m%d_%H%M%S')
    print('LIVE RUN_ID=',os.environ['RUN_ID'],'-> EVAL',tag,'@',os.path.relpath(adapter_dir,'/content') ,flush=True)
    cmd=[sys.executable,'scripts/kg1_colab_realtime_runner.py','--','python','scripts/evaluate_lora_adapter.py',
         '--solution-csv',SOL,'--adapter',adapter_dir,'--output-dir',out,'--label',tag]
    subprocess.run(cmd)

# ===== BASE 086 (adapter na RAIZ) = PISO 0.86 =====
base=snapshot_download(repo_id='felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086',
                       revision='f4134a6d223249d27be2f1c5d94ed59d118d1ce5')
base_dir=base if os.path.exists(base+'/adapter_config.json') else (find_adapter_dirs(base) or [base])[0]
run_eval('086', base_dir)

# ===== CANDIDATO (adapter em runs/<RUN_ID>/{final,checkpoint-N}) =====
CAND_REPO=os.environ.get('CAND_REPO','felipesp1983/kg1-v1244-cot-candidate')
try:
    cand=snapshot_download(repo_id=CAND_REPO, revision=(os.environ.get('CAND_REV') or None))
except Exception as e:
    print('SKIP CANDIDATO',CAND_REPO,'(',str(e)[:80],') -> ainda nao existe? rode o treino antes.',flush=True); cand=None
if cand:
    dirs=find_adapter_dirs(cand)
    rid=os.environ.get('CAND_RUN_ID','')                      # opcional: filtra um RUN_ID especifico
    if rid: dirs=[d for d in dirs if rid in d]
    # ordem: final primeiro, depois checkpoints do mais treinado p/ o menos (160,120,80,40) -> para cedo se final ja bom
    def key(d):
        b=os.path.basename(d)
        if b=='final': return (0,0)
        n=int(b.split('-')[-1]) if b.startswith('checkpoint-') and b.split('-')[-1].isdigit() else 0
        return (1,-n)
    dirs=sorted(dirs,key=key)
    print('Adapters do candidato:',[os.path.relpath(d,cand) for d in dirs],flush=True)
    print('ATENCAO: cada eval ~40-60min. Pare (Ctrl) quando achar o melhor.',flush=True)
    for d in dirs:
        run_eval('CAND_'+os.path.basename(d.rstrip('/')), d)
print('Compare /content/eval_086 vs /content/eval_CAND_* (per_task.csv): bit>=, eq>=, protegidas==, total>823.',flush=True)
